# 1. Azure Identity Concepts & Tokens

Before we call APIs, let's get the vocabulary straight. Azure's identity world has *way* too many terms and most tutorials assume you already know them. We'll learn them by poking at a live mock Entra ID server.

## The four things you need to know

| Term | What it really is | Real-world analogy |
|------|-------------------|--------------------|
| **Tenant** | An Entra directory — contains users, groups, app registrations. Has a GUID. | A company building |
| **App registration** | The *definition* of an application (client ID, secrets, what scopes it exposes). | A job posting describing a role |
| **Service principal** | An *instance* of that app inside a tenant. What actually holds permissions. | A specific person hired for that role |
| **Access token** | A signed JWT proving the caller's identity + permissions. | A badge you swipe at the door |

> In a single tenant, every app registration has one service principal in the same tenant. For multi-tenant apps, admins in other tenants "consent" and a service principal for your app is created in *their* tenant. They control what permissions you get there.

## Setup

```bash
cd enterprise-patterns/azure-authentication
docker compose up -d
```

Select the `Azure Auth (Python)` kernel (top-right). If it's missing, reload the window (`Cmd+Shift+P` → **Reload Window**).

In [ ]:
import httpx, json, base64

AUTHORITY = 'http://localhost:9000/contoso'
TOKEN_URL = f'{AUTHORITY}/oauth2/v2.0/token'

# The OIDC discovery document - every identity provider publishes one.
# SDKs call this to learn the token endpoint, JWKS URI, supported grants.
d = httpx.get(f'{AUTHORITY}/v2.0/.well-known/openid-configuration').json()
print(json.dumps(d, indent=2))

**Why this matters:** when you configure a Python SDK or FastAPI JWT validator, you point it at the issuer URL and it fetches this document automatically. In real Entra ID the URL is:

```
https://login.microsoftonline.com/<tenant-id>/v2.0/.well-known/openid-configuration
```

## App registrations — what's inside one?

Look at `fake-entra/apps.json` — each entry mirrors what you'd see on the **App registrations → overview** blade in the Azure portal:

- `client_id` — the public identifier. Like a username for the app.
- `client_secret` — a password. Alternative: upload a certificate.
- `identifier_uri` (a.k.a. *Application ID URI*) — how *other* apps address this one when requesting a token. Set on the **Expose an API** blade.
- `exposed_scopes` — delegated permissions (user acts through the app): `Files.Read`, `Mail.Send`.
- `app_roles` — application permissions (app acts on its own): `Files.Read.All`.
- `granted_app_roles` — what app-roles *this* app has been granted on *other* apps. In Azure this is configured under **API permissions** and requires admin consent.

In [ ]:
print(json.dumps(json.load(open('../fake-entra/apps.json')), indent=2))

## Anatomy of an access token (JWT)

An access token is three base64url-encoded pieces separated by dots: `header.payload.signature`.

- **header**: algorithm (RS256), key ID (`kid`) — tells validators which public key to use.
- **payload**: the claims — issuer (`iss`), audience (`aud`), subject (`sub`), expiry (`exp`), plus custom claims like `scp` (scopes) or `roles` (app roles).
- **signature**: RSA signature over header+payload. Validators verify it with the public key from JWKS.

Let's get one and look at it.

In [ ]:
r = httpx.post(TOKEN_URL, data={
    'grant_type': 'client_credentials',
    'client_id': 'daemon-client-id',
    'client_secret': 'daemon-secret-value',
    'scope': 'api://api-b/.default',
})
token = r.json()['access_token']
print(token[:60] + '...')

In [ ]:
def decode_jwt(tok):
    header_b64, payload_b64, _sig = tok.split('.')
    pad = lambda s: s + '=' * (-len(s) % 4)
    return {
        'header': json.loads(base64.urlsafe_b64decode(pad(header_b64))),
        'payload': json.loads(base64.urlsafe_b64decode(pad(payload_b64))),
    }

print(json.dumps(decode_jwt(token), indent=2))

### Key claims to know

| Claim | Meaning | Why validators check it |
|-------|---------|-------------------------|
| `iss` | Issuer URL | Is this token from *our* Entra tenant? |
| `aud` | Audience (the API this token is *for*) | Prevents tokens issued for one API being replayed at another |
| `exp` | Expiry (unix seconds) | Short-lived (usually 1 hour) limits damage if stolen |
| `sub` / `oid` | Who the token is about | `oid` is the stable object ID in Entra |
| `azp` | Authorized party — the *client* that requested the token | Useful for logging |
| `scp` | Space-separated **delegated** scopes | User-signed-in flows |
| `roles` | Array of **application** roles | App-only flows |
| `tid` | Tenant ID | Multi-tenant apps use this |

## JWKS — how validators trust the signature

Your API never has a shared secret with Entra. Instead Entra publishes its **public keys** at a well-known JWKS URL. Your API fetches + caches them and verifies the signature.

In [ ]:
jwks = httpx.get(d['jwks_uri']).json()
print(json.dumps(jwks, indent=2))
print('\nToken header kid:', decode_jwt(token)['header']['kid'])
print('JWKS key kid    :', jwks['keys'][0]['kid'])

The `kid` in the token header matches the `kid` in JWKS — that's how the validator picks the right public key when Entra rotates keys.

## What's next

- **Notebook 2** — use this token to call `api-b` (client credentials / S2S).
- **Notebook 3** — *managed identities*: how Azure gives your container a token with no secrets in code.
- **Notebook 4** — *On-Behalf-Of*: user → api-a → api-b with user identity preserved end-to-end.
- **Notebook 5** — local dev with `DefaultAzureCredential`.